### CHVD

In [ ]:
%%bash
# download and extract CHVD sequences
wget https://zenodo.org/records/4498884/files/CHVD_clustered_mash99_v1.tar.gz?download=1 -O chvd_mash99.tar.gz
tar -xzf chvd_mash99.tar.gz

# split chvd into smaller chunks
seqkit split --by-size 10000 CHVD_clustered_mash99_v1.fasta --out-dir chvd_split -j 4

# manually made samplesheet with the following structure
# sample,biosample,acc,source_db,release,fasta
# chvd_split_1,,,CHVD,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/chvd/chvd_split/CHVD_clustered_mash99_v1.part_001.fasta
# chvd_split_2,,,CHVD,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/chvd/chvd_split/CHVD_clustered_mash99_v1.part_002.fasta

### CnGVC

In [ ]:
%%bash
# Download & extract CnGVC sequences
wget https://zenodo.org/records/14671177/files/cnGVC.vOTUs.fa.tar.gz?download=1 -O cnGVC.vOTUs.fa.tar.gz
tar -xzf cnGVC.vOTUs.fa.tar.gz

# split into smaller chunks
seqkit split --by-size 10000 cnGVC.fna --out-dir cngvd_split -j 4

# manually made samplesheet with the following structure
# sample,biosample,acc,source_db,release,fasta
# cngvc_split_1,,,CNGVC,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/cngvc/cngvd_split/cnGVC.part_001.fna
# cngvc_split_2,,,CNGVC,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/cngvc/cngvd_split/cnGVC.part_002.fna

### ENA (MGnify)

In [ ]:
# download metadata for all ENA metagenome assemblies
!curl -X 'POST' \
'https://www.ebi.ac.uk/ena/portal/api/search' -H 'accept: */*' -H 'Content-Type: application/x-www-form-urlencoded' \
-d 'excludeAccessionType=&download=false&query=&excludeAccessions=&includeMetagenomes=true&dataPortal=metagenome&searchCurations=false&includeAccessionType=&includeAccessions=&format=tsv&fields=analysis_accession%2Csample_accession%2Crun_accession%2Cscientific_name%2Cbroad_scale_environmental_context%2Cenvironment_biome%2Chost_body_site%2Cgenerated_ftp%2Cfirst_public&dccDataOnly=false&rule=&result=analysis' \
> ena_metagenomes.tsv

# load ena metagenome metadata
import polars as pl

ena_meta = pl.read_csv('ena_metagenomes.tsv', separator='\t')

# manually look through top organisms to find likely oral samples
ena_meta.group_by('scientific_name').len().sort('len', descending=True).write_csv('ena_metagenomes_top_organisms.csv')

# filter to likely oral samples
ena_oral_samples = (
    ena_meta
        .filter(
            (pl.col('scientific_name').str.contains('oral')) |
            (pl.col('scientific_name').str.contains('nasopharyngeal')) |
            (pl.col('scientific_name').str.contains('lung')) |
            (pl.col('scientific_name').str.contains('saliva '))
        )
)

# create samplesheet for mining oral ena metagenomes
(
    ena_oral_samples
        .rename({
            'analysis_accession':'sample',
            'sample_accession':'biosample',
            'run_accession':'acc'
        })
        .with_columns([
            pl.lit('ENA').alias('source_db'),
            pl.lit('r2025_06').alias('release'),
            (pl.lit('https://') + pl.col('generated_ftp')).alias('fasta'),
        ])
        .drop('generated_ftp')
        .write_csv('ena_oral_samplesheet.csv')
)

### IMG/VR4

In [ ]:
# metadata downloaded from:
# https://genome.jgi.doe.gov/portal/pages/dynamicOrganismDownload.jsf?organism=IMG_VR#

import polars as pl

# load IMG/VR metadata
imgvr_meta = pl.read_csv('/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/phage_database_completeness/IMGVR_all_Sequence_information.tsv',
    separator='\t', columns=['UVIG', 'Ecosystem classification'])

# filter to human-associated UVIGs
(
    imgvr_meta
        .filter(
            (pl.col('Ecosystem classification').str.contains('Human'))
        )[['UVIG']].write_csv('IMGVR_human_uvigs.csv')
)

# download IMG/VR4 sequences
!wget https://portal.nersc.gov/genomad/__data__/IMGVR_DATA/IMGVR4_SEQUENCES.fna

# filter to human-associated UVIGs
!seqkit grep --pattern-file IMGVR_human_uvigs.csv IMGVR4_SEQUENCES.fna --id-regexp "^(.*?)\|" -j 4 --out-file IMGVR4_human_sequences.fna.gz

# split IMG/VR4 human sequences into smaller chunks
!seqkit split --by-size 10000 IMGVR4_human_sequences.fna.gz --out-dir IMGVR4_human_sequences_split -j 4

# manually made samplesheet with the following structure
# sample,biosample,acc,source_db,release,fasta
# imgvr_split_1,,,IMGVR,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/imgvr/IMGVR4_human_sequences_split/IMGVR4_human_sequences.part_001.fna.gz
# imgvr_split_2,,,IMGVR,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/imgvr/IMGVR4_human_sequences_split/IMGVR4_human_sequences.part_002.fna.gz

### Logan

In [ ]:
import duckdb

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

In [ ]:
%%sql
INSTALL httpfs;
LOAD httpfs;
INSTALL parquet;
LOAD parquet;
COPY (
  SELECT 
    acc, biosample, consent
    bioproject, librarysource, libraryselection, librarylayout,
    platform, instrument, organism, mbases, avgspotlen, assay_type
  FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
  WHERE assay_type != 'AMPLICON'
  AND consent = 'public'
  AND libraryselection != 'PCR'
  AND (librarysource = 'METAGENOMIC'
    OR organism LIKE '%microbiom%'
    OR organism LIKE '%metagenom%')
) TO '2025_05_29_sra_metadata.parquet' (FORMAT 'parquet'); 

In [ ]:
import polars as pl

# load SRA metagenome metadata
sra_meta = (
    pl.read_parquet('2025_05_29_sra_metadata.parquet',
        columns=['acc', 'biosample', 'organism', 'librarylayout', 'platform', 'mbases', 'avgspotlen']
    )
    .with_columns([
        pl.col('mbases').cast(pl.Float64),
        pl.col('avgspotlen').cast(pl.Float64),
    ])
)

# write out top organisms for manual inspection
sra_meta.group_by('organism').len().sort('len', descending=True).head(1000).write_csv('sra_metadata_top_organisms.csv')
# top oral related organisms:
# human oral metagenome
# human nasopharyngeal metagenome
# oral metagenome
# human lung metagenome
# respiratory tract metagenome
# human saliva metagenome
# upper respiratory tract metagenome
# human sputum metagenome
# lung metagenome
# human tracheal metagenome
# oral-nasopharyngeal metagenome

# filter to oral related organisms
oral_organisms = set([
    'human oral metagenome',
    'human nasopharyngeal metagenome',
    'oral metagenome',
    'human lung metagenome',
    'respiratory tract metagenome',
    'human saliva metagenome',
    'upper respiratory tract metagenome',
    'human sputum metagenome',
    'lung metagenome',
    'human tracheal metagenome',
    'oral-nasopharyngeal metagenome'
])

sra_meta_oral = (
    sra_meta
        .filter(
            (pl.col('organism').is_in(oral_organisms))
        )
)

# create samplesheet for mining oral logan metagenomes
(
    sra_meta_oral
        .with_columns([
            pl.lit('LOGAN').alias('source_db'),
            pl.lit('r2025_06').alias('release'),
            (pl.lit('s3://logan-pub/c/') + pl.col('acc') + '/' + pl.col('acc') + '.contigs.fa.zst').alias('fasta'),
            pl.lit(None).alias('tar')
        ])
        .write_csv('logan_oral_samplesheet.csv')
)

### MMGE

In [ ]:
%%bash
# downloaded and extract fasta file
wget https://mai.fudan.edu.cn/mgedb/client/file/all_mge_seq.zip
unzip all_mge_seq.zip

# split into smaller chunks
seqkit split --by-size 10000 all_mge_seq.fa --out-dir mmge_split -j 4

# manually made samplesheet with the following structure
# sample,biosample,acc,source_db,release,fasta
# mmge_split_1,,,MMGE,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/mmge/mmge_split/mmge_minlen1500.part_001.fasta.gz
# mmge_split_2,,,MMGE,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/mmge/mmge_split/mmge_minlen1500.part_002.fasta.gz

### OVD

In [ ]:
%%bash

# download OVD sequences
wget https://github.com/grc145/Temp/raw/refs/heads/master/OVD-genomes.fa.bz2

# split OVD sequences into smaller chunks
seqkit split --by-size 10000 OVD-genomes.fa.bz2 --out-dir ovd_split -j 4

# manually made samplesheet with the following structure
# sample,biosample,acc,source_db,release,fasta
# ovd_split_1,,,OVD,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/ovd/ovd_split/OVD-genomes.fa.part_001.bz2
# ovd_split_2,,,OVD,r2025_08,/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/analyses/uhvdb/mine/ovd/ovd_split/OVD-genomes.fa.part_002.bz2

### SPIRE

In [ ]:
import polars as pl

# load spire microontoloy
spire_ont = (
    pl.read_csv("https://swifter.embl.de/~fullam/spire/metadata/spire_v1_microntology.tsv.gz", separator="\t", has_header=False)
    .with_columns(
        pl.col("column_1").str.splitn(by=" ", n=3)
        .struct.rename_fields(["biosample", "bioproject", "ontology"])
        .alias("fields")
    ).unnest("fields")
)

# identify oral samples
spire_oral = (
    spire_ont
        .filter(
            (pl.col("ontology").str.contains("animal host")) &
            (
                (pl.col("ontology").str.contains("mouth")) |
                (pl.col("ontology").str.contains("airway"))
            )
        )
)

# create samplesheet for mining oral SPIRE metagenomes
(
    spire_oral
        .unique(pl.col('biosample'))
        .with_columns([
            pl.lit(None).alias('acc'),
            pl.lit('SPIRE').alias('source_db'),
            pl.lit('r2025_06').alias('release'),
            (pl.lit('https://spire.embl.de/download_assembly/') + pl.col('biosample')).alias('fasta'),
            pl.lit(None).alias('tar'),
            pl.col('biosample').alias('sample'),
        ])
        .drop('bioproject', 'ontology', 'column_1')
).write_csv('spire_oral_samplesheet.csv')